In [ ]:
import kagglehub


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm
# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")
df_Q3 = pd.read_csv(csv_path)



In [ ]:
# Task 2: Write your code here:Inspect the first few rows using head()
df_Q3.head()

In [ ]:
# Task 3: Write your code here:
df_Q3.info()

In [ ]:
# Task 4: Write your code here:
df_Q3.describe()

In [ ]:
# Task 1: Write your code here:
df_Q3.isnull().sum()


In [ ]:
def check_missing_values(df):
  missing_values = df_Q3.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_Q3)

In [ ]:
# Task 2: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df_Q3.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_Q3.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

In [ ]:
# Task 3: Write your code here:
df_Q3['Target'] = df_Q3['Target'].map({'Default': 1, ' No Default': 0})


In [ ]:
# Task 4: Write your code here:
categorical_cols = df_Q3.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_Q3[col] = le.fit_transform(df_Q3[col])
  label_encoders[col] = le

df_Q3

In [ ]:
# Task 5: Write your code here:

print("\nValue counts for the target variable 'Target':")
class_distribution = df_Q3['Target'].value_counts()
print(class_distribution)
plt.figure(figsize=(6, 4))
sns.countplot(data=df_Q3, x='Target')
plt.title('Distribution of Target Variable (Target)')
plt.xlabel('Target')
plt.ylabel('Class')
plt.show()

In [ ]:
# Task 1: Write your code here:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

print("Separating target variable and features...")
X = df_Q3.drop('Target', axis=1)
y = df_Q3['Target']

print("Applying Label Encoding to target variable 'y'...")
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Applying One-Hot Encoding to feature DataFrame 'X'...")
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = pd.DataFrame(onehot_encoder.fit_transform(X), columns=onehot_encoder.get_feature_names_out(X.columns))

print("Verification of encoded data shapes:")
print(f"Shape of X_encoded: {X_encoded.shape}")
print(f"Shape of y_encoded: {y_encoded.shape}")

print("First 5 rows of X_encoded:")
display(X_encoded.head())
print("First 5 elements of y_encoded:")
print(y_encoded[:5])


In [ ]:
# Task 2,3,4,5: Write your code here:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
print("Splitting data into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.3, random_state=42)

print("Data split successful.")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print("Defining custom NumPy implementations for Softmax, Categorical Cross-Entropy, and One-Hot Encoding...")

def softmax(z):
    """Compute softmax values for each set of scores in z."""
    # Subtract max for numerical stability
    e_z = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return e_z / np.sum(e_z, axis=-1, keepdims=True)

def categorical_cross_entropy(y_true, y_pred):
    """Compute categorical cross-entropy loss."""
    epsilon = 1e-10 # Small value to prevent log(0)
    y_pred = np.clip(y_pred, epsilon, 1. - epsilon)
    loss = -np.sum(y_true * np.log(y_pred), axis=-1)
    return np.mean(loss)

def one_hot_encode(labels, num_classes):
    """Convert integer labels to one-hot encoded format."""
    one_hot = np.zeros((len(labels), num_classes))
    one_hot[np.arange(len(labels)), labels] = 1
    return one_hot

print("Functions defined. Testing with example data...")

# Example usage:
# Test Softmax
z_test = np.array([[1.0, 2.0, 3.0], [0.5, 1.5, 2.5]])
softmax_output = softmax(z_test)
print(f"\nSoftmax Output:\n{softmax_output}")
print(f"Softmax Row Sums (should be ~1): {np.sum(softmax_output, axis=1)}")

# Test One-Hot Encode
labels_test = np.array([0, 2, 1, 0])
num_classes_test = 3
one_hot_output = one_hot_encode(labels_test, num_classes_test)
print(f"\nOne-Hot Encoding Output:\n{one_hot_output}")

# Test Categorical Cross-Entropy
y_true_test = one_hot_encode(np.array([0, 1, 2]), 3) # True labels
y_pred_test = np.array([[0.8, 0.1, 0.1], [0.1, 0.7, 0.2], [0.1, 0.2, 0.7]]) # Predicted probabilities
loss_output = categorical_cross_entropy(y_true_test, y_pred_test)
print(f"\nCategorical Cross-Entropy Loss: {loss_output}")

print("Custom functions implemented and tested.")

print("Initializing models and K-Fold cross-validation...")

# Determine the number of classes for one-hot encoding
num_classes = len(np.unique(y_encoded))

# 1. Initialize the machine learning models
models = {
    'Logistic Regression': LogisticRegression(solver='liblinear', max_iter=200, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42) # probability=True is needed for .predict_proba
}

# 2. Create a KFold object
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 3. Create an empty dictionary to store average losses
model_losses = {}

print("Starting K-Fold Cross-Validation for each model...")

# 4. For each model:
for model_name, model in models.items():
    print(f"\nTraining and evaluating {model_name}...")
    fold_losses = [] # List to store loss from each fold

    for fold, (train_index, val_index) in enumerate(kf.split(X_train)):
        # Split X_train and y_train into training and validation sets for the current fold
        X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

        # Train the current model
        model.fit(X_train_fold, y_train_fold)

        # Generate predictions (probabilities) on the validation set
        y_pred_proba = model.predict_proba(X_val_fold)

        # Convert y_val_fold (true labels) into a one-hot encoded format
        y_val_one_hot = one_hot_encode(y_val_fold, num_classes)

        # Calculate categorical cross-entropy loss for the current fold
        loss = categorical_cross_entropy(y_val_one_hot, y_pred_proba)
        fold_losses.append(loss)

    # Calculate the average loss for the model
    avg_loss = np.mean(fold_losses)
    model_losses[model_name] = avg_loss

    # Print the average cross-validation loss for the current model
    print(f"{model_name} - Average Cross-Validation Loss: {avg_loss:.4f}")

print("\nAll models trained and evaluated. Stored average losses:")
print(model_losses)
print("Evaluating models on the test set...")

# Dictionary to store performance metrics
model_performance = {}

# Iterate through each trained model
for model_name, model in models.items():
    print(f"\nEvaluating {model_name}...")

    # Make predictions on the test set (hard labels)
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    # Store metrics
    model_performance[model_name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

    # Print metrics
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")

    # Generate and visualize Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title(f'Confusion Matrix for {model_name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

print("\nAll models evaluated.")

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: